___
# FIGURE 3BCD *Process* inpatient pain decoding
Shirvalkar et al., Nature Medicine 2026

This notebook processes pain decoding data (P3 for Figure 3D and remaining participants for completeness).
Feature importance per brain region is extracted using linear discriminant analysis (LDA)
and least absolute shrinkage and selection operator (LASSO) models across pain metrics and time windows.

Steps:
1. Load model coefficients (LDA & LASSO).
2. Load bootstrap distributions and identify significant models per pain metric.
3. Calculate region-level feature importance from significant model coefficients.
4. Order feature counts by electrode anatomy for downstream plotting.

Inputs:
- main_dir: directory of repository so files can be loaded/saved:
    - `{P#}_feature_names.xlsx`                : List of features for the participant
    - `{P#}_lda_across_models.xlsx`             : LDA model coefficients
    - `{P#}_lasso_across_models.xlsx`            : LASSO model coefficients
    - `{P#}_lda_bootstrap_distribution.xlsx`     : LDA bootstrap performance
    - `{P#}_lasso_bootstrap_distribution.xlsx`   : LASSO bootstrap performance
    - `{P#}_electrode_labels.xlsx`               : Electrode names and anatomy

Outputs:
- `{P#}_lda_across_models.xlsx`       : LDA dataframe updated with significance
- `{P#}_lasso_across_models.xlsx`     : LASSO dataframe updated with significance
- `{P#}_inpatient_feature_counts.xlsx` : brain region-level (collapsed across electrodes) feature importance counts
    - `P3_inpatient_feature_counts.xlsx` provides the counts shown in Figure 3D.
___


### Repository root (resolved automatically; set `NATMED_REPO` to override)

In [15]:
import os
# Repository root: defaults to the parent of python_code/, i.e. this notebook's
# grandparent folder. Override by setting the NATMED_REPO environment variable
# or replacing this line with an explicit path to your local copy.
main_dir = os.environ.get('NATMED_REPO',
    os.path.abspath(os.path.join(os.getcwd(), os.pardir))) + '/'


In [2]:
### could adjust these parameters, but keep as is to reproduce Shirvalkar et al 2026
# Participants to process
pt_labels      = ['P1' , 'P2', 'P3', 'P4', 'P5']
# Pain metrics to analyze
SELECT_METRICS = ['intensity_vas_s0', 'nrs_s0', 'unpleasantness_vas_s0', 'relief_s0']

In [3]:
# import modules
import numpy as np
import pandas as pd
from pathlib import Path
import sys
sys.path.append(str(Path(main_dir) / 'python_code/shared_utilities'))
from utility.figure3_inpatient_decoding_helpers import (identify_significant_models, process_metric_coefficients)
# set data directory based off of main_dir
data_dir = Path(main_dir) / 'data/figure3_processed_inpatient_pain_decoding'

## MAIN PROCESSING LOOP PER particpant
### load LDA/Lasso models
### compare R^{2} & AUC, respectively to bootstrap distribution
### count & then order which brain regions are in top 10 features

In [6]:
for ind_pt, pt in enumerate(pt_labels):
    print(f"\n{'='*60}\nPROCESSING {pt}\n{'='*60}")

    # Load feature names
    feature_names = pd.read_excel(Path(data_dir) / f"{pt}_feature_names.xlsx").T.values[0]

    # Helper to load coefficient array
    def load_coef_array(df_path):
        df = pd.read_excel(df_path)
        coef_array = np.vstack(df["coef"].apply(lambda x: np.fromstring(x.strip("[]"), sep=' ')))
        return df, coef_array

    lda_df, lda_coef_array     = load_coef_array(Path(data_dir) / f"{pt}_lda_across_models.xlsx")
    lasso_df, lasso_coef_array = load_coef_array(Path(data_dir) / f"{pt}_lasso_across_models.xlsx")

    # Identify significant models
    def load_and_identify(df, boots_path,metric):
        boots = pd.read_excel(boots_path).to_numpy()
        df = identify_significant_models(df, boots, metric)
        return df, sum(df.sig_indices)

    lda_df, lda_sig_count     = load_and_identify(lda_df, Path(data_dir) / f"{pt}_lda_bootstrap_distribution.xlsx", metric='auc')
    lasso_df, lasso_sig_count = load_and_identify(lasso_df, Path(data_dir) / f"{pt}_lasso_bootstrap_distribution.xlsx",  metric='r-squared')
   
    # save LDA and Lasso across models——NOW with p-value from bootstrap distribution
    lda_df.to_excel(Path(data_dir) / f"{pt}_lda_across_models.xlsx", index=False)
    lasso_df.to_excel(Path(data_dir) / f"{pt}_lasso_across_models.xlsx", index=False)

    # Compute feature importance counts per metric
    def compute_feature_counts(coef_array, df, model_type):
        all_counts = []
        for metric in SELECT_METRICS:
            print(f"  Processing metric: {metric} ({model_type})")
            counts = process_metric_coefficients(coef_array, df, feature_names, metric, model_type)
            if counts is not None:
                all_counts.append(counts)
        return pd.concat(all_counts).groupby(level=0).sum() if all_counts else pd.DataFrame()

    lda_combined   = compute_feature_counts(lda_coef_array, lda_df, "LDA")
    lasso_combined = compute_feature_counts(lasso_coef_array, lasso_df, "LASSO")

    # Order counts by electrode position
    electrode_df = pd.read_excel(Path(data_dir) / f"{pt}_electrode_labels.xlsx")

    def order_counts(combined_counts):
        return [combined_counts.loc[chl, 'count'] if chl in combined_counts.index else 0
                for chl in electrode_df['elec_numbered']]

    lda_counts_ordered   = order_counts(lda_combined)
    lasso_counts_ordered = order_counts(lasso_combined)

    feature_counts = pd.DataFrame({
                                    'anatomy':        electrode_df['elec_anatomy'].to_numpy(),
                                    'count_lda':      lda_counts_ordered,
                                    'count_lasso':    lasso_counts_ordered,
                                    'count_combined':  np.array(lda_counts_ordered) + np.array(lasso_counts_ordered)
                                   }, index=electrode_df['elec_numbered'])

    # All participants (including P3, the Figure 3D example) are saved to one folder.
    save_dir = Path(main_dir) / 'data' / 'figure3_processed_inpatient_pain_decoding' / f"{pt}_inpatient_feature_counts.xlsx"
    feature_counts.to_excel(save_dir, index=False)
    print(f"  Saved channel-ordered counts for {len(electrode_df)} electrodes")


PROCESSING P1
  Processing metric: intensity_vas_s0 (LDA)
    LDA significant models for intensity_vas_s0: 4
  Processing metric: nrs_s0 (LDA)
    LDA significant models for nrs_s0: 3
  Processing metric: unpleasantness_vas_s0 (LDA)
    LDA significant models for unpleasantness_vas_s0: 3
  Processing metric: relief_s0 (LDA)
    LDA significant models for relief_s0: 1
  Processing metric: intensity_vas_s0 (LASSO)
    LASSO significant models for intensity_vas_s0: 2
  Processing metric: nrs_s0 (LASSO)
    LASSO significant models for nrs_s0: 3
  Processing metric: unpleasantness_vas_s0 (LASSO)
    LASSO significant models for unpleasantness_vas_s0: 1
  Processing metric: relief_s0 (LASSO)
    No significant LASSO models for relief_s0
  Saved channel-ordered counts for 51 electrodes

PROCESSING P2
  Processing metric: intensity_vas_s0 (LDA)
    LDA significant models for intensity_vas_s0: 1
  Processing metric: nrs_s0 (LDA)
    LDA significant models for nrs_s0: 2
  Processing metric: un